In [ ]:
# ==================== INSTALL REQUIRED LIBRARIES ====================
!pip install datasets transformers scikit-learn pandas numpy matplotlib nltk tensorflow gensim --quiet

# ==================== IMPORTS ====================
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import joblib
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# ==================== NLTK DOWNLOADS ====================
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# ==================== LOAD DATASET ====================
dataset = load_dataset("takala/financial_phrasebank", "sentences_allagree")
df = pd.DataFrame(dataset['train'])

# ==================== PREPROCESSING ====================
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = re.findall(r'\b\w+\b', text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['sentence'].apply(preprocess)

# ==================== ENCODE LABELS ====================
label_encoder = LabelEncoder()
df['encoded_label'] = label_encoder.fit_transform(df['label'])

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['encoded_label'], test_size=0.2, random_state=42
)

# ==================== 1. LOGISTIC REGRESSION ====================
print("\nTraining Logistic Regression...")

tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

log_model = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, penalty='l2')
log_model.fit(X_train_tfidf, y_train)
y_pred_log = log_model.predict(X_test_tfidf)

print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log))

joblib.dump(log_model, "logistic_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

# ==================== 2. ENHANCED LSTM ====================
print("\nTraining Enhanced LSTM...")

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

max_len = 120
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=max_len),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)),
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3)

model.fit(X_train_pad, y_train, epochs=25, batch_size=64,
          validation_split=0.1, callbacks=[early_stop, reduce_lr])

y_pred_lstm = np.argmax(model.predict(X_test_pad), axis=-1)

print("\nEnhanced LSTM Classification Report:")
print(classification_report(y_test, y_pred_lstm))

model.save("enhanced_lstm_model.h5")
joblib.dump(tokenizer, "lstm_tokenizer.pkl")

# ==================== 3. FINBERT ====================
print("\nTraining FinBERT...")

finbert_tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
finbert_model = AutoModelForSequenceClassification.from_pretrained(
    "yiyanghkust/finbert-tone", num_labels=3
)

def tokenize_fn(example):
    return finbert_tokenizer(example["sentence"], padding="max_length", truncation=True)

encoded_dataset = dataset.map(tokenize_fn, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_test = encoded_dataset["train"].train_test_split(test_size=0.2, seed=42)
train_ds = train_test["train"]
test_ds = train_test["test"]

training_args = TrainingArguments(
    output_dir="./finbert_checkpoints",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=finbert_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=finbert_tokenizer
)

trainer.train()

finbert_model.save_pretrained("finbert_model")
finbert_tokenizer.save_pretrained("finbert_model")

print("\nFinBERT trained and saved successfully.")

# ==================== 4. WORD2VEC + LSTM ====================
print("\nTraining LSTM with Word2Vec Embeddings...")

tokenized_texts = [text.split() for text in df['clean_text']]

w2v_model = Word2Vec(sentences=tokenized_texts, vector_size=100, window=5, min_count=2, workers=4)
w2v_model.save("word2vec.model")

tokenizer_w2v = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer_w2v.fit_on_texts(df['clean_text'])

word_index = tokenizer_w2v.word_index
embedding_dim = 100
embedding_matrix = np.zeros((10000, embedding_dim))

for word, i in word_index.items():
    if i < 10000 and word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

X_train_seq_w2v = tokenizer_w2v.texts_to_sequences(X_train)
X_test_seq_w2v = tokenizer_w2v.texts_to_sequences(X_test)

X_train_pad_w2v = pad_sequences(X_train_seq_w2v, maxlen=max_len, padding='post')
X_test_pad_w2v = pad_sequences(X_test_seq_w2v, maxlen=max_len, padding='post')

w2v_lstm_model = Sequential([
    Embedding(input_dim=10000, output_dim=embedding_dim, weights=[embedding_matrix],
              input_length=max_len, trainable=False),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)),
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])

w2v_lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])

w2v_lstm_model.fit(X_train_pad_w2v, y_train, epochs=25, batch_size=64,
                   validation_split=0.1, callbacks=[early_stop, reduce_lr])

y_pred_w2v_lstm = np.argmax(w2v_lstm_model.predict(X_test_pad_w2v), axis=-1)

print("\nWord2Vec + LSTM Classification Report:")
print(classification_report(y_test, y_pred_w2v_lstm))

w2v_lstm_model.save("word2vec_lstm_model.h5")
joblib.dump(tokenizer_w2v, "word2vec_tokenizer.pkl")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.2.1 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 20.0.0 which is incompatible.
pylibcudf-cu12 25.2.1 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 20.0.0 which is incompatible.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

financial_phrasebank.py:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

The repository for takala/financial_phrasebank contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/takala/financial_phrasebank.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


...

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


26/26 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.5634 - loss: 1.0365 - val_accuracy: 0.6319 - val_loss: 0.9273 - learning_rate: 5.0000e-04
Epoch 2/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.6233 - loss: 0.9381 - val_accuracy: 0.6484 - val_loss: 0.8982 - learning_rate: 5.0000e-04
Epoch 3/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.6260 - loss: 0.9061 - val_accuracy: 0.6429 - val_loss: 0.8618 - learning_rate: 5.0000e-04
Epoch 4/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.6182 - loss: 0.8929 - val_accuracy: 0.6319 - val_loss: 0.8526 - learning_rate: 5.0000e-04
Epoch 5/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - accuracy: 0.6256 - loss: 0.8838 - val_accuracy: 0.6209 - val_loss: 0.8415 - learning_rate: 5.0000e-04
Epoch 6/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.6254 - loss: 0.8541 - val_accuracy: 0.6209 - val_loss: 0.8295 - learning_rate: 5.0000e-04
Epoch 7/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 42s 1s/step - accuracy: 0.6463 - loss: 0.8525 - v

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Word2Vec + LSTM Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        56
           1       0.78      0.93      0.85       276
           2       0.52      0.54      0.53       121

    accuracy                           0.71       453
   macro avg       0.43      0.49      0.46       453
weighted avg       0.61      0.71      0.66       453



['word2vec_tokenizer.pkl']

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

# Load FinBERT tokenizer and model from the local path
model_path = "./finbert_model"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)


In [ ]:
import torch
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification

# Convert Series to list of strings
X_test = X_test.astype(str).tolist()

# Tokenize
inputs = tokenizer(X_test, return_tensors='pt', padding=True, truncation=True, max_length=128)

# Move model and inputs to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Predict
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    finbert_preds = torch.argmax(logits, dim=1).cpu().numpy()

# Calculate and print accuracy
finbert_accuracy = accuracy_score(y_test, finbert_preds)
print(f"FinBERT Test Accuracy: {finbert_accuracy:.4f}")


FinBERT Test Accuracy: 0.8808
